# EMT Ph3 Four-Terminal V-Type SSN: Coupled Inductor / Transformer Test

This notebook tests the newly implemented `GenericFourTerminalVTypeSSN` with a transformer-like coupled inductor model.

The four terminals are interpreted as:

```text
terminal 0 = primary top
terminal 2 = primary bottom

terminal 1 = secondary top
terminal 3 = secondary bottom
```

The internal SSN model is:

```text
        primary winding                     secondary winding

 t0 o──── Rp,Lp ────o t2          t1 o──── Rs,Ls ────o t3
          iP --->                         iS --->

                 magnetic coupling M between both windings
```

The voltage equations are:

```text
vP = v0 - v2 = Rp iP + Lp diP/dt + M  diS/dt

vS = v1 - v3 = Rs iS + M  diP/dt + Ls diS/dt
```

State vector:

```text
x = [iP_abc;
     iS_abc]
```

Input vector:

```text
u = [v0_abc;
     v1_abc;
     v2_abc;
     v3_abc]
```

Output vector:

```text
y = [i0_abc;
     i1_abc;
     i2_abc;
     i3_abc]
```

The confirmed dynamic current convention for this four-terminal V-type implementation is used:

```text
branch current t0 -> t2: terminal 0 = -iP, terminal 2 = +iP
branch current t1 -> t3: terminal 1 = -iS, terminal 3 = +iS
```

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import dpsimpy as dpsim

print([x for x in dir(dpsim.emt.ph3) if "FourTerminal" in x])

## Simulation parameters

In [ ]:
frequency = 50.0
time_step = 50e-6
final_time = 0.05

source_voltage = 1.0 * np.exp(1j * 0.0)

source_resistance = dpsim.Math.single_phase_parameter_to_three_phase(1.0)
load_resistance = dpsim.Math.single_phase_parameter_to_three_phase(50.0)

primary_resistance = 2.0 * np.eye(3)
secondary_resistance = 2.0 * np.eye(3)

primary_inductance = 20e-3 * np.eye(3)
secondary_inductance = 20e-3 * np.eye(3)

k_coupling = 0.5
mutual_inductance = k_coupling * np.sqrt(20e-3 * 20e-3) * np.eye(3)

load_resistance_value = 50.0
weak_reference_resistance_value = 1e6

## State-space matrices for the four-terminal transformer SSN

In [ ]:
def create_four_terminal_transformer_matrices():
    i3 = np.eye(3)

    lp = primary_inductance
    ls = secondary_inductance
    m = mutual_inductance

    rp = primary_resistance
    rs = secondary_resistance

    lmat = np.zeros((6, 6))
    lmat[0:3, 0:3] = lp
    lmat[0:3, 3:6] = m
    lmat[3:6, 0:3] = m
    lmat[3:6, 3:6] = ls

    print("eig(lmat):", np.linalg.eigvals(lmat))
    print("cond(lmat):", np.linalg.cond(lmat))

    if np.linalg.cond(lmat) > 1e8:
        raise ValueError("Inductance matrix is ill-conditioned.")

    inv_lmat = np.linalg.inv(lmat)

    rmat = np.zeros((6, 6))
    rmat[0:3, 0:3] = rp
    rmat[3:6, 3:6] = rs

    # x = [iP; iS]
    # u = [vin; vout] = [v2 - v0; v3 - v1]
    #
    # vP = vin
    # vS = vout

    a = -inv_lmat @ rmat
    b = inv_lmat

    # y = [iin; iout]
    # Let interface currents equal winding currents.
    c = np.eye(6)

    d = np.zeros((6, 6))

    print("A,B,C,D:", a.shape, b.shape, c.shape, d.shape)

    return a, b, c, d

## Run four-terminal transformer SSN simulation

In [ ]:
def run_four_terminal_transformer_ssn():
    sim_name = "PY_EMT_Ph3_FourTerminalVTypeSSN_Transformer"

    n_src = dpsim.emt.SimNode("nSrc", dpsim.PhaseType.ABC)
    n_p_top = dpsim.emt.SimNode("nPTop", dpsim.PhaseType.ABC)
    n_s_top = dpsim.emt.SimNode("nSTop", dpsim.PhaseType.ABC)
    n_p_bottom = dpsim.emt.SimNode("nPBottom", dpsim.PhaseType.ABC)
    n_s_bottom = dpsim.emt.SimNode("nSBottom", dpsim.PhaseType.ABC)

    a, b, c, d = create_four_terminal_transformer_matrices()

    vs = dpsim.emt.ph3.VoltageSource("VS")
    vs.set_parameters(
        dpsim.Math.single_phase_variable_to_three_phase(source_voltage),
        frequency,
    )

    r_source = dpsim.emt.ph3.Resistor("RSource")
    r_source.set_parameters(source_resistance)

    trafo_ssn = dpsim.emt.ph3.GenericFourTerminalVTypeSSN("TransformerSSN")
    trafo_ssn.set_parameters(a, b, c, d)

    load = dpsim.emt.ph3.Resistor("Load")
    load.set_parameters(load_resistance)

    vs.connect([dpsim.emt.SimNode.gnd, n_src])
    r_source.connect([n_src, n_p_top])
    trafo_ssn.connect([n_p_top, n_s_top, n_p_bottom, n_s_bottom])
    load.connect([n_s_top, n_s_bottom])

    r_primary_ground = dpsim.emt.ph3.Resistor("RPrimaryGround")
    r_primary_ground.set_parameters(
        dpsim.Math.single_phase_parameter_to_three_phase(1e-3)
    )
    r_primary_ground.connect([n_p_bottom, dpsim.emt.SimNode.gnd])

    r_secondary_reference = dpsim.emt.ph3.Resistor("RSecondaryReference")
    r_secondary_reference.set_parameters(
        dpsim.Math.single_phase_parameter_to_three_phase(1e9)
    )
    r_secondary_reference.connect([n_s_bottom, dpsim.emt.SimNode.gnd])

    system = dpsim.SystemTopology(
        frequency,
        [n_src, n_p_top, n_s_top, n_p_bottom, n_s_bottom],
        [vs, r_source, trafo_ssn, load, r_primary_ground, r_secondary_reference],
    )

    i_intf = trafo_ssn.attr("i_intf")
    print("i_intf type:", type(i_intf))
    print("i_intf dir:", dir(i_intf))

    dpsim.Logger.set_log_dir("logs/" + sim_name)
    logger = dpsim.Logger(sim_name)

    logger.log_attribute("vSrc", n_src.attr("v"))
    logger.log_attribute("iTransformer", trafo_ssn.attr("i_intf"))
    logger.log_attribute("vTransformer", trafo_ssn.attr("v_intf"))
    logger.log_attribute("vPTop", n_p_top.attr("v"))
    logger.log_attribute("vPBottom", n_p_bottom.attr("v"))
    logger.log_attribute("vSTop", n_s_top.attr("v"))
    logger.log_attribute("vSBottom", n_s_bottom.attr("v"))
    logger.log_attribute("iLoad", load.attr("i_intf"))

    sim = dpsim.Simulation(sim_name)
    sim.set_system(system)
    sim.add_logger(logger)
    sim.set_domain(dpsim.Domain.EMT)
    sim.set_time_step(time_step)
    sim.set_final_time(final_time)
    sim.run()

    return sim_name

## Run simulation

In [ ]:
sim_name = run_four_terminal_transformer_ssn()

print("Log:", Path("logs") / sim_name / f"{sim_name}.csv")

## Load logs

In [ ]:
def load_log(sim_name):
    path = Path("logs") / sim_name / f"{sim_name}.csv"
    df = pd.read_csv(path, skipinitialspace=True)
    df.columns = df.columns.str.strip()
    return df


log = load_log(sim_name)

display(log.head())

print("Columns:")
print(list(log.columns))

In [ ]:
# Numerical assertions for SSN transformer test

required_columns = [
    "time",
    "vPTop_0",
    "vPBottom_0",
    "vSTop_0",
    "vSBottom_0",
    "iLoad_0",
    "iLoad_1",
    "iLoad_2",
]

for col in required_columns:
    assert col in log.columns, f"Missing expected log column: {col}"

assert np.all(
    np.isfinite(log[required_columns].to_numpy())
), "Log contains NaN or Inf values."

# Check that simulation did not trivially stay at zero
v_primary = log["vPTop_0"] - log["vPBottom_0"]
v_secondary = log["vSTop_0"] - log["vSBottom_0"]

assert np.max(np.abs(v_primary)) > 1e-6, "Primary voltage is almost zero."
assert np.max(np.abs(v_secondary)) > 1e-6, "Secondary voltage is almost zero."
assert np.max(np.abs(log["iLoad_0"])) > 1e-9, "Load current is almost zero."

print("Numerical SSN assertions passed.")

# Skip startup transient
steady = slice(len(log) // 2, None)

vp_rms = np.sqrt(np.mean(np.abs(v_primary.iloc[steady]) ** 2))
vs_rms = np.sqrt(np.mean(np.abs(v_secondary.iloc[steady]) ** 2))

ratio = vp_rms / vs_rms

expected_ratio = 2  # replace with your transformer ratio

print(f"Measured voltage ratio: {ratio}")
print(f"Expected voltage ratio: {expected_ratio}")

assert np.isclose(
    ratio,
    expected_ratio,
    rtol=0.1,  # 10% tolerance
), (
    f"Transformer ratio mismatch: " f"measured={ratio}, expected={expected_ratio}"
)

## Plot helpers

In [ ]:
def first_matching_column(df, prefix):
    matches = [c for c in df.columns if c.startswith(prefix)]
    if not matches:
        matches = [c for c in df.columns if prefix in c]
    if not matches:
        raise KeyError(
            f"No column matching {prefix!r}. Available columns: {list(df.columns)}"
        )
    return matches[0]


def time_column(df):
    for candidate in ["time", "Time", "t"]:
        if candidate in df.columns:
            return candidate
    return df.columns[0]

## Primary and secondary voltages

In [ ]:
t = time_column(log)

vp_top = first_matching_column(log, "vPTop")
vp_bottom = first_matching_column(log, "vPBottom")
vs_top = first_matching_column(log, "vSTop")
vs_bottom = first_matching_column(log, "vSBottom")

v_primary = log[vp_top] - log[vp_bottom]
v_secondary = log[vs_top] - log[vs_bottom]

plt.figure()
plt.plot(log[t], v_primary, label="Primary voltage")
plt.plot(log[t], v_secondary, "--", label="Secondary voltage")
plt.xlabel("Time [s]")
plt.ylabel("Voltage")
plt.title("Transformer SSN primary and secondary voltages")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

log.columns = [c.strip() for c in log.columns]

v_primary = log["vPTop_0"] - log["vPBottom_0"]
v_secondary = log["vSTop_0"] - log["vSBottom_0"]

plt.figure(figsize=(8, 4))
plt.plot(log["time"], v_primary, label="vPTop - vPBottom")
plt.plot(log["time"], v_secondary, label="vSTop - vSBottom")
plt.xlabel("Time [s]")
plt.ylabel("Voltage")
plt.title("Physical transformer winding voltages")
plt.grid(True)
plt.legend()
plt.show()

## Load current

In [ ]:
plt.figure(figsize=(8, 4))

for col in [c for c in log.columns if "iLoad" in c]:
    plt.plot(log["time"], log[col], label=col)

plt.xlabel("Time [s]")
plt.ylabel("Current")
plt.title("Load currents")
plt.grid(True)
plt.legend()
plt.show()

## Interface current components

In [ ]:
itr_cols = [c for c in log.columns if c.startswith("iTransformer")]

print("Transformer interface current columns:")
print(itr_cols)

if itr_cols:
    plt.figure()
    for col in itr_cols[:6]:
        plt.plot(log[t], log[col], label=col)
    plt.xlabel("Time [s]")
    plt.ylabel("Current")
    plt.title("First transformer interface current components")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
log.columns = [c.strip() for c in log.columns]

# Load current through secondary load
i_load_a = log["iLoad_0"]

# If iTransformer logs only one scalar, fix logger first.
# It should ideally log 6 components: iTransformer_0 ... iTransformer_5
itr_cols = [c for c in log.columns if c.startswith("iTransformer")]
print(itr_cols)

In [ ]:
print([c for c in log.columns if "Transformer" in c])
print(log.columns.tolist())

log.columns = [c.strip() for c in log.columns]

plt.figure(figsize=(8, 4))
plt.plot(log["time"], log["iLoad_0"], label="iLoad phase A")
plt.plot(log["time"], log["iLoad_1"], label="iLoad phase B")
plt.plot(log["time"], log["iLoad_2"], label="iLoad phase C")
plt.xlabel("Time [s]")
plt.ylabel("Current [A]")
plt.title("Secondary load currents")
plt.grid(True)
plt.legend()
plt.show()

## Notes

If this diverges, the first things to check are:

1. Reduce the coupling coefficient, e.g. `coupling_coefficient = 0.5`.
2. Increase winding resistances.
3. Check whether the secondary island needs a stronger reference resistor.
4. Confirm that the inductance matrix is positive definite.

The inductance matrix must satisfy:

```text
M^2 < Lp Ls
```

which is why the notebook uses:

```python
M = k * sqrt(Lp * Ls), with k < 1
```